# Traceprop review runs

**Pull-and-run.** Re-running the clone cell does a `git pull` (public repo, no token), so it
always picks up the latest fixes. Run on an **A100**.

Current status:
- **Job 1 (exp36 memory/storage) — LIVE.** Fixed to track the headline 34-module scope; the
  cell prints the count, which must read `34`.
- **Jobs 2-4 — DEFERRED** (items 2 and 8). Left here for a future pinned-environment rerun;
  do not run them for the current submission (the first run came back in the wrong dtype/stack).
- **Item 9 (rank agreement) — pending.** Its cell is added once the generator is tiny/CPU-tested.

After Job 1, download `exp36_memory_storage_pythia1b.json` from `MyDrive/traceprop_runs/` into
your local `experiments/results/` and send it back.

In [ ]:
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU-only'
print('GPU:', name)
if 'A100' not in name:
    print('WARNING: Job 1 (exp36) needs an A100 for the peak-memory measurement.')

!pip -q install datasets scipy scikit-learn accelerate
!pip -q uninstall -y torchao 2>/dev/null
!pip -q install --ignore-requires-python logix-ai
import numpy, transformers, peft
print('numpy', numpy.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/traceprop_runs', exist_ok=True)


In [ ]:
# Public repo: clone if absent, else pull. Re-run this cell to pick up latest fixes.
import os
if os.path.isdir('/content/Traceprop/.git'):
    !cd /content/Traceprop && git pull -q
else:
    !git clone -q https://github.com/AmitoVrito/Traceprop.git /content/Traceprop
%cd /content/Traceprop
!git log --oneline -1
!pip -q install -e .
%cd /content/Traceprop/experiments
!mkdir -p results


## Job 1 (LIVE) — memory/storage, item 1. A100, ~5 min.

Confirm the first printed line reads `tracked modules under headline scope: 34`.

In [ ]:
!python exp36_memory_storage.py --model EleutherAI/pythia-1b --device cuda \
    --steps 200 --batch 8 --seq 128 --kfac 8 --track 0 \
    --out results/exp36_memory_storage_pythia1b.json
!cp results/exp36_memory_storage_pythia1b.json /content/drive/MyDrive/traceprop_runs/ 2>/dev/null
print('copied exp36 json to Drive')


## Job 5 (LIVE) — item 9 rank agreement at 1B. A100 80GB, ~15 min.

CPU-verified on the tiny backend (dot Spearman 0.77, preconditioned 0.89, delta-stable). This is the 1B run: N=128 test queries against the FULL headline train set (n_train=1600 = 200 steps x batch 8). Memory is not the limiter on 80GB; time is, so the LogIX damping grid is trimmed to the 3 needed deltas. If it overruns ~15 min, drop --n_test (e.g. 64), keep n_train=1600. Only the small `_report.json` is needed back.

In [ ]:
!cd /content/Traceprop && git pull -q && git log --oneline -1
%cd /content/Traceprop/experiments
# n_train=1600 = full headline train set (200 steps x batch 8). LogIX grid trimmed to the
# 3 needed deltas (--precond_damping_grid) so --tune_logix does 3 scoring passes, not 10.
!python exp35_logix_lds.py --backend hf --model EleutherAI/pythia-1b --data synthetic \
    --device cuda --skip_gpu_check \
    --factored --inline_precond --lora_init random --tune_logix --logix_permodule_damping \
    --n_train 1600 --n_test 128 --epochs 2 \
    --precond_damping_grid 1e-4,1e-3,1e-2 --rank_deltas 1e-3,1e-4,1e-2 \
    --rank_agreement_out results/exp39_rank_agreement_1b.npz
!cp results/exp39_rank_agreement_1b_report.json /content/drive/MyDrive/traceprop_runs/ 2>/dev/null
print('copied rank-agreement report to Drive')


## Job 2 (DEFERRED, item 2) — 6.9B split.

Do not run for this submission: on an 80GB A100 `--dtype auto` picked fp32, not the
paper's bf16. Revisit only in a pinned environment with `--dtype bf16` forced.

In [ ]:
# DEFERRED -- left for a pinned-environment rerun, not for the current submission.
# !python exp35_inlineprecond_overhead_hf.py --model EleutherAI/pythia-6.9b \
#     --device cuda --dtype bf16 --track 0 --kfac 8 --steps 50 --batch 8 --seq 128 \
#     --repeats 5 --with_logix --skip_logix_pca \
#     --out results/exp35_inlineprecond_overhead_hf_pythia7b_split.json


## Job 3 (DEFERRED, item 8 overhead) — 1B k-sweep overhead.

In [ ]:
# DEFERRED.
# for k in [8, 32, 64]:
#     !python exp35_inlineprecond_overhead_hf.py --model EleutherAI/pythia-1b \
#         --device cuda --dtype fp32 --track 0 --kfac {k} --steps 200 --batch 8 \
#         --seq 128 --repeats 10 --with_logix --out results/exp35_overhead_1b_k{k}.json


## Job 4 (DEFERRED, item 8 quality) — tiny-model k-sweep LDS.

Errored on the first run (full-backward-hook issue under the newer torch); needs a fix.

In [ ]:
# DEFERRED.
# for s in range(5):
#     !python exp35_logix_lds.py --backend tiny --data synthetic --device cpu \
#         --track 0 --factored --inline_precond --tune_logix --lora_init random \
#         --kfac_values 8,32,64 --seed {s} --out results/exp35_ksweep_seed{s}.npz
